### Retrival Augmentented Generation Day 2

In [1]:
%pip install langchain_cohere -q
%pip install spacy -q
%pip install psycopg2 -q
%pip install python-dotenv -q
#ignore error

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.1.4 which is incompatible.
transformers 4.31.0 requires tokenizers!=0.11.3,<0.14,>=0.11.1, but you have tokenizers 0.19.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [46]:
# now you need to run this in a terminal window
# python -m spacy download en_core_web_md
# now restart your kernel

Standard imports for the libraires we will be using in this notebook.  Try to keep your imports in the first cell so this can this code can more easliy be converted into a python program later

In [2]:
import boto3
import pandas as pd
import json
import time
import os
import numpy as np
import pyarrow
import traceback
from langchain.embeddings import BedrockEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import BedrockChat
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import BedrockEmbeddings
import dbconnection
import psycopg2
from psycopg2 import OperationalError
from dotenv import load_dotenv

load_dotenv()
# Create the AWS client for the Bedrock runtime with boto3
aws_client = boto3.client(service_name="bedrock-runtime")

#### Lets define functions that will use various embedding models so we can generate vector embeddings
Spacey

In [3]:
def generate_spacy_vector_embedding(text):
    embedder = SpacyEmbeddings(model_name="en_core_web_md")
    query_embedding = embedder.embed_query(text)

    return(np.array(query_embedding))

Cohere

In [4]:
# send in an array size of one and only return the 0th element
def generate_cohere_vector_embedding(text_data):
    input_type = "clustering"
    truncate = "NONE" # optional
    model_id = "cohere.embed-english-v3" # or "cohere.embed-multilingual-v3"
    
    # Create the JSON payload for the request
    json_params = {
            'texts': [text_data],
            'truncate': truncate, 
            "input_type": input_type
        }
    json_body = json.dumps(json_params)
    params = {'body': json_body, 'modelId': model_id,}
    
    # Invoke the model and print the response
    result = aws_client.invoke_model(**params)
    response = json.loads(result['body'].read().decode())
    return(np.array(response['embeddings'][0]))


Amazon Titan

In [5]:
# Let's generate a dense vector using Amazon Titan with LangChain
def generate_titan_vector_embedding(text):
    #create an Amazon Titan Text Embeddings client
    embeddings_client = BedrockEmbeddings(region_name="us-west-2") 

    #Invoke the model
    embedding = embeddings_client.embed_query(text)
    return(np.array(embedding))



In [6]:
# Let's generate a dense vector using Amazon Titan without using a np.array as a return value
def generate_vector_embedding(text):
    #create an Amazon Titan Text Embeddings client
    embeddings_client = BedrockEmbeddings(region_name="us-west-2") 

    #Invoke the model
    embedding = embeddings_client.embed_query(text)
    #Note pgvector does not want a np.array as out manual method
    return(embedding)



This is the mathmatical formula to calcuate cosine similarity between 2 vectors

In [7]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    similarity = dot_product / (norm_vec1 * norm_vec2)
    return similarity



In [8]:
def clean_value(value):
    value_str = str(value)
    cleaned_value = ''.join(char for char in value_str if char.isalnum() or char.isspace())
    return cleaned_value

In [9]:
def limit_string_size(x, max_chars=2048):
    # Check if the input is a string
    if isinstance(x, str):
        return x[:max_chars]
    else:
        return x

In [11]:
def print_top_values(list_stuff: list, num_items: int) -> None:
    i=0
    for item in list_stuff:
        i=i+1
        if i>num_items:
            return None
        print(item)

In [56]:
# clean abstract text
#df = pd.read_csv('data/latest_research_articles.csv')
#df['abstract'] = df['abstract'].apply(clean_value)

#df
dft = pd.read_pickle('data/embedded_df.pkl')

In [12]:
# Generate HyDE context

def generate_hyde_response(query_phrase):
    model_id = "anthropic.claude-3-sonnet-20240229-v1:0"
    # Each model will named parameters which will likely be different depending on the providor
    model_kwargs =  { 
        "max_tokens": 400, # This is the maximum output tokens you want the model to use
        "temperature": 1,  # Temperature controls the randomness and creativity of the generated text.
        "top_k": 250,      # Top-k parameter determines the number of highest probability next word choices the model should conside
        "top_p": 0.9,      # Top-p sampling considers the cumulative probability distribution of the next word choices and sets a probability threshold
        "stop_sequences": ["\n\nHuman"],
    }
    # LangChain tooling
    model = BedrockChat(
        client=aws_client,
        model_id=model_id,
        model_kwargs=model_kwargs,
    )
    
    human_prompt = "Given the following question \n {query} can you please generate a paragraph of text that answers the question. Be sure to use scientific \
                    medical terminology. Please just include the paragraph in your response."
    # Uses the messaging method which is required for all Claude 3 calls
    messages = [
        ("system", "You are a helpful assistant"),
        ("human", human_prompt),
    ]
    try:
        prompt = ChatPromptTemplate.from_messages(messages)
        # LangChain at work
        chain = prompt | model | StrOutputParser()


        # Send the message content to Claude using Bedrock and get the response
        start_time = time.time()  # Start timing
        # Call Bedrock
        response = chain.invoke({"query": query_phrase})
        end_time = time.time()  # End timing
        print("Claude call took :", end_time - start_time)  # Calculate execution time

        return(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = traceback.sys.exc_info()
        line_number = exc_traceback.tb_lineno
        print(f"Errort: {exc_type}{exc_value}{exc_traceback} on {line_number}")

In [13]:
def calculate_zscores(cosine_scores):
    zscores = []
    # Calculate the mean of the sample points
    mean = np.mean(cosine_scores)
    # Calculate the standard deviation of the sample points
    std_deviation = np.std(cosine_scores, ddof=1)  # ddof=1 for sample standard deviation
    # Calculate the z-scores for each sample point
    z_scores = [(x - mean) / std_deviation for x in cosine_scores]

    return z_scores

In [14]:
# Now let's take the records that are greater that 1/2 the top Z-score and send those to the LLM for an answer
def best_answer(data, question):
    model_id = "anthropic.claude-3-sonnet-20240229-v1:0"

    model_kwargs =  { 
        "max_tokens": 2048,
        "temperature": 0.0,
        "top_k": 250,
        "top_p": 0.9,
        "stop_sequences": ["\n\nHuman"],
    }

    model = BedrockChat(
        client=aws_client,
        model_id=model_id,
        model_kwargs=model_kwargs,
    )

    human_prompt = "You are to answer the question using the data in the following information.  Do not make up your answer, only use \
                    supporting data from the article, If you don't have enough data simply respond, I don't have enough information to answer that question. \
                    given the following article data {data} can you please give a concise answer to the following question. {question}"
    messages = [
        ("system", "You are a helpful assistant that can answer quesitons based on news articles you have been given."),
        ("human", human_prompt),
    ]
    try:
        prompt = ChatPromptTemplate.from_messages(messages)

        chain = prompt | model | StrOutputParser()

        # Chain Invoke
        
    
        # Send the message content to Claude using Bedrock and get the response
        start_time = time.time()  # Start timing
        # Call Bedrock
        response = chain.invoke({"data": data,"question": question})
        end_time = time.time()  # End timing
        #print("Claude call took :", end_time - start_time)  # Calculate execution time

        return(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = traceback.sys.exc_info()
        line_number = exc_traceback.tb_lineno

        return f"ERROR generating good answer: {exc_type}{exc_value}{exc_traceback} on {line_number}"


In [60]:
from requests import get

ip = get('https://api.ipify.org').content.decode('utf8')
print('My public IP address is: {}'.format(ip))


My public IP address is: 54.203.17.143


In [61]:
# Fill in your username so yoru database entry will be unique to you
#MY_USERNAME = "dkraker@calpoly.edu"
MY_USERNAME = "YOUR USERNAME HERE"

Let's define a function that will put data into our vector DB

In [62]:
def insert_record_into_db(table_name, record, conn):

    # Dynamically generates and executes an INSERT SQL statement for PostgreSQL, handling
    # special data types like datetime objects and arrays directly.
    
    # Args:
    # - table_name (str): The name of the table into which the record will be inserted.
    # - data (dict): A dictionary representing the record to be inserted, where keys are column names
    #                  and values can include native PostgreSQL types like datetime and arrays.
    # - conn (psycopg2.connection): A psycopg2 connection object.
    
    # Generate column names and placeholders
    columns = ', '.join(record.keys())
    placeholders = ', '.join(['%s'] * len(record))  # PostgreSQL uses %s as placeholder

    # Create the INSERT INTO statement
    sql = f'INSERT INTO {table_name} ({columns}) VALUES ({placeholders})'
    #print("columns=", columns)
    #print("placeholders=", placeholders)
    #print(sql)
    try:
        cur = conn.cursor()
        
        cur.execute(sql, tuple(record.values()))
        conn.commit()
        print("Record inserted successfully.")
    except psycopg2.Error as e:
        print("An error occurred:", e)


In [63]:
# Load your large text document
with open("data/staff-report.txt", "r") as file:
    text = file.read()

# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,  # Adjust the chunk size as needed
    chunk_overlap=512,  # Adjust the overlap between chunks as needed
)

# Split the text into chunks
chunks = text_splitter.split_text(text)

conn = dbconnection.open_connection_to_db()
try:
    # create a dataframe with new chunk raw text
    for chunk in chunks:
        data_record = {}
        v_embed = generate_vector_embedding(chunk)
        data_record["username"] = MY_USERNAME
        data_record["textattribute1"] = chunk
        data_record["textattribute2"] = ""
        data_record["textattribute3"] = ""
        data_record["textattribute4"] = ""
        data_record["textattribute5"] = ""
        data_record["textembedding1"] = v_embed
        insert_record_into_db("rag", data_record, conn)        
        
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    if conn:
        conn.close()

Error connecting to database: connection to server on socket "/tmp/.s.PGSQL.5432" failed: No such file or directory
	Is the server running locally and accepting connections on that socket?

An error occurred: 'NoneType' object has no attribute 'cursor'


Now let's use the database as a way to find the best match. Notice the 2 different commented SQL syntax differences.  We can easily compute Euclidean distance as well

In [64]:
def run_similarity_search_pgvector(question, embedded_text, conn):
    # Cosine similarity
    #1-(textembedding1 <=> ('{embedded_text}')) as cosine_similar  \
    # Euclidean distance
    #textembedding1 <-> ('{embedded_text}') as euclidean_distance  \
    sql = f"SELECT textattribute1, textattribute2, textattribute3, textattribute4, textattribute5, \
                1-(textembedding1 <=> ('{embedded_text}')) as cosine_similar  \
                FROM public.rag \
                WHERE username = '{MY_USERNAME}' \
                ORDER BY cosine_similar DESC \
                LIMIT 50"
    #print(sql)
    cosine_scores = []
    try:
        article_text = ""
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()

        # grab the cosine scores so we can compute Z score for narrow article selection
        # need all scores so we can calc Z
        for row in rows:
            #print(row[5])
            cosine_scores.append(row[5])
            
        z_scores = calculate_zscores(cosine_scores)
        answer = "Unknown"
        article_text = ""
        zscore_index = 0
        first_z_score = z_scores[0]
        for row in rows:
            if(first_z_score/2)<z_scores[zscore_index]:
                print(f"Using chunk with a cosine match of: {row[5]} and Z-score of: {z_scores[zscore_index]}")
                article_text = article_text + row[0] + "\n"
            zscore_index += 1
        #print(article_text)
        answer = best_answer(article_text, question)
        #Close cursor and connection
        cur.close()
        return answer
        
    except psycopg2.Error as e:
            print("An error occurred:", e)
    finally:
        if conn:
            conn.close()
        
  

In [65]:
def view_my_data(conn):
    try:
        cur = conn.cursor()

        # SQL statement to delete rows where username is 'bob'
        sql = f"SELECT textattribute1, textattribute2, textattribute3, textattribute4, textattribute5 FROM public.rag WHERE username = '{MY_USERNAME}'"
    
        # Execute the SQL statement
        cur.execute(sql)
        rows = cur.fetchall()

        # grab the cosine scores so we can compute Z score for narrow article selection
        for row in rows:
            print(row[0], row[1], row[2], row[3], row[4])
    
        

        
    except psycopg2.Error as e:
            print("An error occurred:", e)
    finally:
        if conn:
            conn.close()
    

In [66]:
conn = dbconnection.open_connection_to_db()
view_my_data(conn)

Error connecting to database: connection to server on socket "/tmp/.s.PGSQL.5432" failed: No such file or directory
	Is the server running locally and accepting connections on that socket?



AttributeError: 'NoneType' object has no attribute 'cursor'

In [67]:
def purge_my_data(conn):
    
    try:
        cur = conn.cursor()

        # SQL statement to delete rows where username is 'bob'
        sql = f"DELETE FROM public.rag WHERE username = '{MY_USERNAME}'"
    
        # Execute the SQL statement
        cur.execute(sql)
    
        # Commit the changes to the database
        conn.commit()
    
        # Get the number of affected rows
        deleted_rows = cur.rowcount
        print(f"{deleted_rows} row(s) deleted.")

        
    except psycopg2.Error as e:
            print("An error occurred:", e)
    finally:
        if conn:
            conn.close()

In [68]:
conn = dbconnection.open_connection_to_db()
purge_my_data(conn)

Error connecting to database: connection to server on socket "/tmp/.s.PGSQL.5432" failed: No such file or directory
	Is the server running locally and accepting connections on that socket?



AttributeError: 'NoneType' object has no attribute 'cursor'

### RAG Homework 2
Now that you've seen the power of RAG you need to design and build a small application of your own using all the techniques you've learned to answer interesting questions that only your dataset can answer

1) Find a interesting set of information data that is NOT present in Claude 3 or simililar and prep / clean the data for ingestion

Data Source: https://www.kaggle.com/datasets/victornuez/latest-research-articles

In [15]:
# Step 1: Data Preparation and Cleaning
file_path = 'data/latest_research_articles_2024.csv'
data = pd.read_csv(file_path)
data['abstract'] = data['abstract'].apply(clean_value)
# for cohere we need to limit the string size to 2048
#df['abstract'] = df['abstract'].apply(limit_string_size)
data

,title,abstract,doi,citations,accesses,online_attention,published_datetime,tweeters,blogs,facebook_pages,news_outlets,redditors,video_uploaders,wikipedia_page,mendeley,Topic
0,Estimates of the reproduction ratio from epide...,Accurate estimates of the reproduction ratio a...,https://doi.org/10.1038/s41567-024-02471-7,0,0,0,2024-04-25,5,0,0,0,0,0,0,0,Physics
1,Spin Berry curvature-enhanced orbital Zeeman e...,Berry phases and the related concept of Berry ...,https://doi.org/10.1038/s41567-024-02487-z,0,801,1,2024-04-22,1,0,0,0,0,0,0,0,Physics
2,Room-temperature flexible manipulation of the ...,The quantum metric and Berry curvature are two...,https://doi.org/10.1038/s41567-024-02476-2,0,1029,53,2024-04-22,14,0,1,7,0,0,0,0,Physics
3,Irreversible entropy transport enhanced by fer...,The nature of particle and entropy flow betwee...,https://doi.org/10.1038/s41567-024-02483-3,0,636,1,2024-04-22,2,0,0,0,0,0,0,0,Physics
4,Penning-trap measurement of the Q value of ele...,The investigation of the absolute scale of the...,https://doi.org/10.1038/s41567-024-02461-9,0,2025,105,2024-04-19,6,1,0,14,0,0,0,0,Physics
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,Evidence for an odd-parity nematic phase above...,The search for quantum states arising from the...,https://doi.org/10.1038/s41567-023-02272-4,0,3350,16,2024-01-04,11,0,0,1,0,0,0,20,Physics
102,Ultrafast formation of topological defects in ...,Topological defects play a central role in dyn...,https://doi.org/10.1038/s41567-023-02279-x,1,4259,10,2024-01-04,27,0,0,0,0,0,0,20,Physics
103,Guiding Trojan light beams via Lagrange points,The guided transmission of optical waves is cr...,https://doi.org/10.1038/s41567-023-02270-6,1,2697,90,2024-01-03,106,0,0,3,0,0,0,13,Physics
104,Heating and cooling are fundamentally asymmetr...,According to conventional wisdom a system plac...,https://doi.org/10.1038/s41567-023-02269-z,1,22k,154,2024-01-03,109,0,0,12,1,0,0,32,Physics


2) Ingest this data into a data store for later retrieval.  1 extra credit point if you choose to use the class vector DB provided

In [16]:
# Apply the embedding function to the cleaned review texts
#data['embedded_abstract'] = data['abstract'].apply(generate_titan_vector_embedding)

# Save the dataframe with embeddings for later use
#data.to_pickle('data/LatestResearchArticles2024_Embedded.pkl')

dft = pd.read_pickle('data/LatestResearchArticles2024_Embedded.pkl')

In [18]:
### Retrieval from embedded sources
#Now that we have a dataframe with embedded content of interest, we can use semantic similarity to retrieve the right data to feed to an LLM

# Given the following query let's generate context that more closely matches the embedded data
#query = "What is the latest research about quantum physics"
#query = "As light from a star spreads out and weakens, do gaps form between the photons"
query = "What is quantum error correction"

In [20]:
# Step 2: Generate Embeddings

query_vector = generate_titan_vector_embedding(generate_hyde_response(query))
# This is a tuple of the article index and the cosine similarity score
results = []
# Iterate over each row in the DataFrame
for index, row in dft.iterrows():
    # Extract the value from the specified column
    article_embedding = row['embedded_abstract']
    results.append((index, cosine_similarity(article_embedding, query_vector)))
    #print (index, value)

results.sort(key=lambda x: x[1], reverse=True)
i = 0
# Print the sorted data
print("Here are a few articles that may match your interest:")
for item in results:
    # Use the index from the Original dataframe to extract values of interest
    article_title = dft.iloc[item[0]]['title']
    print(f"Abtract: '{article_title}' with a cosine match of: {item[1]}")
    i=i+1
    if i == 5:
        break

Claude call took : 3.882730722427368
Here are a few articles that may match your interest:
Abtract: 'Protecting expressive circuits with a quantum error detection code' with a cosine match of: 0.7453461505619009
Abtract: 'Time-Efficient Constant-Space-Overhead Fault-Tolerant Quantum Computation' with a cosine match of: 0.7094379421140817
Abtract: 'Protecting entanglement between logical qubits via quantum error correction' with a cosine match of: 0.6554351498522244
Abtract: 'Programmable Heisenberg interactions between Floquet qubits' with a cosine match of: 0.491128900461596
Abtract: 'Emergence of highly coherent two-level systems in a noisy and dense quantum network' with a cosine match of: 0.4586358673632495


3. Choose a retrieval method to respond to input queries and using either an approach demonstrated in class or one of your own that reduces the retrieval context length 

In [21]:
# Find Cosine Similarity Scores
results = []
for index, row in dft.iterrows():
    article_embedding = row['embedded_abstract']
    similarity_score = cosine_similarity(article_embedding, query_vector)
    results.append((index, similarity_score))

# Sort Results by Similarity Scores
results.sort(key=lambda x: x[1], reverse=True)

# Calculate Z-Scores
cosine_scores = [item[1] for item in results]
z_scores = calculate_zscores(cosine_scores)

In [22]:
# Step 3: Implement Retrieval System

articles = ""
i = 0
j = 0
first_z_score = z_scores[0]

print("Here are a few articles that may match your interest:")
for item, z_score in zip(results, z_scores):
    article_title = dft.iloc[item[0]]['title']
    abstract = dft.iloc[item[0]]['abstract']
    # Only include if z-score is above the threshold
    if (first_z_score / 2) < z_score:
        print(f"Abstract: '{article_title}' with a cosine match of: {item[1]} and Z-score of: {z_score}")
        articles += " " + abstract
        j += 1
    i += 1
    if j == 5:  # Limit the number of articles retrieved
        break

print(f"Grabbed the top {j} scores")

Here are a few articles that may match your interest:
Abstract: 'Protecting expressive circuits with a quantum error detection code' with a cosine match of: 0.7453461505619009 and Z-score of: 4.168331979177137
Abstract: 'Time-Efficient Constant-Space-Overhead Fault-Tolerant Quantum Computation' with a cosine match of: 0.7094379421140817 and Z-score of: 3.89750336381542
Abstract: 'Protecting entanglement between logical qubits via quantum error correction' with a cosine match of: 0.6554351498522244 and Z-score of: 3.4902009088572825
Abstract: 'Programmable Heisenberg interactions between Floquet qubits' with a cosine match of: 0.491128900461596 and Z-score of: 2.250962418794339
Grabbed the top 4 scores


4) Use an LLM of your choosing to provide a concise answer to the question asked.  Ensure the information provided can be referenced in your original source data

In [23]:
# Step 4: RAG Approach to Provide Concise Answers

# Now let's take the records that are greater that 1/2 the top Z-score and send those to the LLM for an answer
def best_answer(data, question):
    model_id = "anthropic.claude-3-sonnet-20240229-v1:0"

    model_kwargs =  { 
        "max_tokens": 2048,
        "temperature": 0.0,    # temperature is 0 because we want exact from the query-no creativity wanted 
        "top_k": 250,
        "top_p": 0.9,
        "stop_sequences": ["\n\nHuman"],
    }

    model = BedrockChat(
        client=aws_client,
        model_id=model_id,
        model_kwargs=model_kwargs,
    )

    human_prompt = "You are to answer the question using the data in the following information.  Do not make up your answer, only use \
                    supporting data from the article, If you don't have enough data simply respond, I don't have enough information to answer that question. \
                    given the following article data {data} can you please give a concise answer to the following question. {question}"
    messages = [
        ("system", "You are a helpful assistant that can answer quesitons based on news articles you have been given."),
        ("human", human_prompt),
    ]
    try:
        prompt = ChatPromptTemplate.from_messages(messages)

        chain = prompt | model | StrOutputParser()

        # Chain Invoke
        
    
        # Send the message content to Claude using Bedrock and get the response
        start_time = time.time()  # Start timing
        # Call Bedrock
        response = chain.invoke({"data": data,"question": question})
        end_time = time.time()  # End timing
        #print("Claude call took :", end_time - start_time)  # Calculate execution time

        return(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = traceback.sys.exc_info()
        line_number = exc_traceback.tb_lineno

        return f"ERROR generating good answer: {exc_type}{exc_value}{exc_traceback} on {line_number}"


In [24]:
print("Here is the best answer I could find about ", query)
print(best_answer(articles, query))

Here is the best answer I could find about  What is quantum error correction
Based on the information provided in the articles, quantum error correction refers to techniques and protocols aimed at protecting quantum information from errors caused by noise and decoherence effects in quantum systems. The key points about quantum error correction from the articles are:

1. A successful quantum error correction protocol would allow quantum computers to run algorithms without suffering from the effects of noise.

2. The articles describe different approaches to implement quantum error correction codes, such as the "Iceberg code" for trapped-ion quantum computers and concatenating smaller quantum codes like Hamming codes.

3. Quantum error correction can enhance the coherence time of entangled logical qubits compared to their unprotected counterparts, as demonstrated in one of the experiments mentioned.

4. Repetitive quantum error correction is used to protect entanglement, which is fragile